In [2]:
using Random
using Statistics
using Printf

# Set a random seed for reproducibility
Random.seed!(0)


TaskLocalRNG()

In [3]:
# Generate the spiral dataset
function spiral_data(samples::Int, classes::Int)
    X = zeros(Float32, samples * classes, 2)
    y = zeros(Int, samples * classes)
    for class_number in 1:classes
        # Julia is 1-indexed, so we adjust the range
        ix = (samples * (class_number - 1) + 1):(samples * class_number)
        r = LinRange(0.0, 1.0, samples) # radius
        t = LinRange((class_number - 1) * 4, class_number * 4, samples) .+ randn(Float32, samples) .* 0.2f0 # theta
        
        # Use element-wise broadcasting with dots (.)
        X[ix, 1] = r .* sin.(t .* 2.5)
        X[ix, 2] = r .* cos.(t .* 2.5)
        y[ix] .= class_number
    end
    return X, y
end

spiral_data (generic function with 1 method)

In [4]:
# --- Dense Layer ---
mutable struct Layer_Dense
    weights::Matrix{Float32}
    biases::Matrix{Float32}
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dweights::Matrix{Float32}
    dbiases::Matrix{Float32}
    dinputs::Matrix{Float32}

    function Layer_Dense(n_inputs::Int, n_neurons::Int)
        weights = 0.01f0 .* randn(Float32, n_inputs, n_neurons)
        biases = zeros(Float32, 1, n_neurons)
        new(weights, biases, Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0),
            Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
    end
end

function forward(layer::Layer_Dense, inputs::Matrix{Float32})
    layer.inputs = inputs
    layer.output = inputs * layer.weights .+ layer.biases # .+ is broadcasted addition
end

function backward(layer::Layer_Dense, dvalues::Matrix{Float32})
    # ' is the transpose operator in Julia
    layer.dweights = layer.inputs' * dvalues
    layer.dbiases = sum(dvalues, dims=1)
    layer.dinputs = dvalues * layer.weights'
end

backward (generic function with 1 method)

In [5]:
# --- ReLU Activation ---
mutable struct Activation_ReLU
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dinputs::Matrix{Float32}
    Activation_ReLU() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(activation::Activation_ReLU, inputs::Matrix{Float32})
    activation.inputs = inputs
    activation.output = max.(0.0f0, inputs) # Element-wise max
end

function backward(activation::Activation_ReLU, dvalues::Matrix{Float32})
    activation.dinputs = copy(dvalues)
    # Use logical indexing to zero out gradients
    activation.dinputs[activation.inputs .<= 0] .= 0.0f0
end


backward (generic function with 2 methods)

In [6]:
# --- Combined Softmax and Loss ---
mutable struct Activation_Softmax_Loss_CategoricalCrossentropy
    output::Matrix{Float32} # Stores Softmax probabilities
    dinputs::Matrix{Float32}
    Activation_Softmax_Loss_CategoricalCrossentropy() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, inputs::Matrix{Float32}, y_true::Vector{Int})
    # Softmax Activation (with numerical stability)
    exp_values = exp.(inputs .- maximum(inputs, dims=2))
    probabilities = exp_values ./ sum(exp_values, dims=2)
    combo.output = probabilities

    # Categorical Cross-Entropy Loss
    n_samples = size(probabilities, 1)
    probs_clipped = clamp.(probabilities, 1f-7, 1 - 1f-7)
    
    # Get probabilities for the true classes (note: Julia is 1-indexed)
    correct_confidences = [probs_clipped[i, y_true[i]] for i in 1:n_samples]
    
    # Calculate and return mean loss
    return mean(-log.(correct_confidences))
end

function backward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, y_true::Vector{Int})
    n_samples = size(combo.output, 1)
    combo.dinputs = copy(combo.output)

    # Simplified gradient: Predicted - GroundTruth
    for (i, true_class_idx) in enumerate(y_true)
        combo.dinputs[i, true_class_idx] -= 1
    end

    # Normalize gradient
    combo.dinputs = combo.dinputs ./ n_samples
end

backward (generic function with 3 methods)

In [7]:
# --- SGD Optimizer ---
mutable struct Optimizer_SGD
    learning_rate::Float64
    Optimizer_SGD(learning_rate=1.0) = new(learning_rate)
end

function update_params(optimizer::Optimizer_SGD, layer::Layer_Dense)
    # Use in-place, broadcasting subtraction (.-=)
    layer.weights .-= optimizer.learning_rate .* layer.dweights
    layer.biases .-= optimizer.learning_rate .* layer.biases
end

update_params (generic function with 1 method)

In [8]:
# 1. Instantiate Components
println("--- Initializing Network and Data ---")
X, y = spiral_data(100, 3)

dense1 = Layer_Dense(2, 64)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(64, 3)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()
optimizer = Optimizer_SGD(1.0)

println("--- Starting Training ---")
# --- Header for the output table ---
@printf "%-10s%-15s%-15s\n" "Epoch" "Accuracy" "Loss"
@printf "%-10s%-15s%-15s\n" "----------" "---------------" "---------------"


# 2. Training Loop (10,001 Epochs)
for epoch in 0:10000
    
    # a. Forward Pass
    forward(dense1, X)
    forward(activation1, dense1.output)
    forward(dense2, activation1.output)
    loss = forward(loss_activation, dense2.output, y)

    # Calculate Accuracy
    # Get prediction indices by finding the max in each row
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)

    # Print progress every 1000 epochs
    if epoch % 1000 == 0
        @printf "%-10d%-15.3f%-15.3f\n" epoch accuracy loss
    end

    # b. Backward Pass
    backward(loss_activation, y)
    backward(dense2, loss_activation.dinputs)
    backward(activation1, dense2.dinputs)
    backward(dense1, activation1.dinputs)
    
    # c. Optimization (Update Parameters)
    update_params(optimizer, dense1)
    update_params(optimizer, dense2)
end

println("-----------------------------------------")
println("Training complete.")


--- Initializing Network and Data ---
--- Starting Training ---
Epoch     Accuracy       Loss           
----------------------------------------
0         0.250          1.099          
1000      0.423          1.045          
2000      0.423          1.043          
3000      0.413          1.040          
4000      0.427          1.038          
5000      0.433          1.036          
6000      0.443          1.034          
7000      0.423          1.031          
8000      0.430          1.029          
9000      0.427          1.030          
10000     0.433          1.032          
-----------------------------------------
Training complete.


In [9]:
#optimizer : Learning Rate Decay
# --- SGD Optimizer ---
mutable struct Optimizer_SGD
    learning_rate::Float64
    Optimizer_SGD(learning_rate=1.0) = new(learning_rate)
end

function update_params(optimizer::Optimizer_SGD, layer::Layer_Dense)
    # Use in-place, broadcasting subtraction (.-=)
    layer.weights .-= optimizer.learning_rate .* layer.dweights
    layer.biases .-= optimizer.learning_rate .* layer.biases
end


update_params (generic function with 1 method)

### --- SGD Optimizer with Learning Rate Decay ---

In [10]:
mutable struct Optimizer_SGD_with_decay
    learning_rate::Float64
    current_learning_rate::Float64

    Decay::Float64
    iterations:Int


    #Constructor to initialize the optimizer
    function Optimizer_SGD(learning_rate::Float64=1.0, decay::Float64=0.0)
        # current_learning_rate starts as the initial learning_rate
        new(learning_rate, learning_rate, decay, 0)
        
    end
end

UndefVarError: UndefVarError: `iterations` not defined in `Main`
Suggestion: check for spelling errors or missing imports.


    pre_update_params(optimizer::Optimizer_SGD)

Calculates the new `current_learning_rate` based on the decay formula.
This should be called ONCE per epoch, before updating any layer parameters.


In [11]:
function pre_update_params(optimizer::Optimizer_SGD_with_decay)
        if optimizer.decay > 0.0
        optimizer.current_learning_rate = optimizer.learning_rate * (1.0 / (1.0 + optimizer.decay * optimizer.iterations))
    end
end


pre_update_params (generic function with 1 method)


    update_params(optimizer::Optimizer_SGD, layer::Layer_Dense)

Updates a single layer's weights and biases using the current learning rate.


In [14]:
function update_params(optimizer::Optimizer_SGD, layer::Layer_Dense)
    # Use the current_learning_rate for the update
    # The '.-=' operator performs in-place, broadcasted subtraction
    layer.weights .-= optimizer.current_learning_rate .* layer.dweights
    layer.biases .-= optimizer.current_learning_rate .* layer.dbiases
end


update_params (generic function with 1 method)

    post_update_params(optimizer::Optimizer_SGD)

Increments the iteration counter.
This should be called ONCE per epoch, after updating all layer parameters.

In [16]:
function post_update_params(optimizer::Optimizer_SGD)
    optimizer.iterations += 1
end
println("Optimizer_SGD with decay functionality defined.")

Optimizer_SGD with decay functionality defined.


In [17]:
using Random
using Statistics
using Printf

# Set seed for reproducibility
Random.seed!(0)

# --- Data Generation and Model Classes (from previous implementation) ---


TaskLocalRNG()

In [18]:
function spiral_data(samples::Int, classes::Int)
    X = zeros(Float32, samples * classes, 2)
    y = zeros(Int, samples * classes)
    for class_number in 1:classes
        ix = (samples * (class_number - 1) + 1):(samples * class_number)
        r = LinRange(0.0f0, 1.0f0, samples)
        t = LinRange((class_number - 1) * 4, class_number * 4, samples) .+ randn(Float32, samples) .* 0.2f0
        X[ix, 1] = r .* sin.(t .* 2.5f0)
        X[ix, 2] = r .* cos.(t .* 2.5f0)
        y[ix] .= class_number
    end
    return X, y
end

spiral_data (generic function with 1 method)

In [19]:
# --- MAIN TRAINING SCRIPT ---
println("\n--- Initializing Pipeline with Decay Optimizer ---")

# 1. Generate Data and Create Model Instances
X, y = spiral_data(100, 3)
dense1 = Layer_Dense(2, 64)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(64, 3)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()


--- Initializing Pipeline with Decay Optimizer ---


Activation_Softmax_Loss_CategoricalCrossentropy(Matrix{Float32}(undef, 0, 0), Matrix{Float32}(undef, 0, 0))

In [22]:
# 2. Instantiate the new optimizer WITH A DECAY RATE
optimizer = Optimizer_SGD_with_decay(learning_rate=1.0, decay=1e-4)

MethodError: MethodError: no method matching Optimizer_SGD_with_decay(; learning_rate::Float64, decay::Float64)
The type `Optimizer_SGD_with_decay` exists, but no method is defined for this combination of argument types when trying to construct it.

## Clean code till decay optimizer

In [1]:

using Random
using Statistics
using Printf

# Set a random seed for reproducibility
Random.seed!(0)

# --- Data Generation Function ---
function spiral_data(samples::Int, classes::Int)
    X = zeros(Float32, samples * classes, 2)
    y = zeros(Int, samples * classes)
    for class_number in 1:classes
        ix = (samples * (class_number - 1) + 1):(samples * class_number)
        r = LinRange(0.0f0, 1.0f0, samples)
        t = LinRange((class_number - 1) * 4, class_number * 4, samples) .+ randn(Float32, samples) .* 0.2f0
        X[ix, 1] = r .* sin.(t .* 2.5f0)
        X[ix, 2] = r .* cos.(t .* 2.5f0)
        y[ix] .= class_number
    end
    return X, y
end

# --- Dense Layer ---
mutable struct Layer_Dense
    weights::Matrix{Float32}
    biases::Matrix{Float32}
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dweights::Matrix{Float32}
    dbiases::Matrix{Float32}
    dinputs::Matrix{Float32}

    function Layer_Dense(n_inputs::Int, n_neurons::Int)
        weights = 0.01f0 .* randn(Float32, n_inputs, n_neurons)
        biases = zeros(Float32, 1, n_neurons)
        new(weights, biases, Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
    end
end

function forward(layer::Layer_Dense, inputs::Matrix{Float32})
    layer.inputs = inputs
    layer.output = inputs * layer.weights .+ layer.biases
end

function backward(layer::Layer_Dense, dvalues::Matrix{Float32})
    layer.dweights = layer.inputs' * dvalues
    layer.dbiases = sum(dvalues, dims=1)
    layer.dinputs = dvalues * layer.weights'
end

# --- ReLU Activation ---
mutable struct Activation_ReLU
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dinputs::Matrix{Float32}

    Activation_ReLU() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(activation::Activation_ReLU, inputs::Matrix{Float32})
    activation.inputs = inputs
    activation.output = max.(0.0f0, inputs)
end

function backward(activation::Activation_ReLU, dvalues::Matrix{Float32})
    activation.dinputs = copy(dvalues)
    activation.dinputs[activation.inputs .<= 0.0f0] .= 0.0f0
end

# --- Combined Softmax and Loss ---
mutable struct Activation_Softmax_Loss_CategoricalCrossentropy
    output::Matrix{Float32}
    dinputs::Matrix{Float32}

    Activation_Softmax_Loss_CategoricalCrossentropy() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, inputs::Matrix{Float32}, y_true::Vector{Int})
    # Softmax activation
    exp_values = exp.(inputs .- maximum(inputs, dims=2))
    combo.output = exp_values ./ sum(exp_values, dims=2)

    # Categorical Cross-Entropy Loss
    n_samples = size(combo.output, 1)
    probs_clipped = clamp.(combo.output, 1e-7, 1 - 1e-7)
    correct_confidences = [probs_clipped[i, y_true[i]] for i in 1:n_samples]
    
    return mean(-log.(correct_confidences))
end

function backward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, y_true::Vector{Int})
    n_samples = size(combo.output, 1)
    combo.dinputs = copy(combo.output)

    for (i, true_class_idx) in enumerate(y_true)
        combo.dinputs[i, true_class_idx] -= 1
    end
    combo.dinputs = combo.dinputs ./ n_samples
end

# --- SGD Optimizer with Learning Rate Decay ---
mutable struct Optimizer_SGD
    learning_rate::Float64
    current_learning_rate::Float64
    decay::Float64
    iterations::Int

    function Optimizer_SGD(learning_rate::Float64=1.0, decay::Float64=0.0)
        new(learning_rate, learning_rate, decay, 0)
    end
end

function pre_update_params(optimizer::Optimizer_SGD)
    if optimizer.decay > 0.0
        optimizer.current_learning_rate = optimizer.learning_rate * (1.0 / (1.0 + optimizer.decay * optimizer.iterations))
    end
end

function update_params(optimizer::Optimizer_SGD, layer::Layer_Dense)
    layer.weights .-= optimizer.current_learning_rate .* layer.dweights
    layer.biases .-= optimizer.current_learning_rate .* layer.dbiases
end

function post_update_params(optimizer::Optimizer_SGD)
    optimizer.iterations += 1
end

# --- Main Training Script ---
function main()
    println("--- Initializing Pipeline ---")
    
    # 1. Generate Data and Create Model Instances
    X, y = spiral_data(100, 3)
    dense1 = Layer_Dense(2, 64)
    activation1 = Activation_ReLU()
    dense2 = Layer_Dense(64, 3)
    loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()
    
    # 2. Instantiate the optimizer with a decay rate
    optimizer = Optimizer_SGD(learning_rate=1.0, decay=1e-4)
    
    println("\n--- Starting Training Loop ---")
    @printf "%-10s %-15s %-15s %-20s\n" "Epoch" "Accuracy" "Loss" "Learning Rate"
    println(repeat("-", 60))

    # 3. Training Loop
    for epoch in 0:10000
        # Forward Pass
        forward(dense1, X)
        forward(activation1, dense1.output)
        forward(dense2, activation1.output)
        loss = forward(loss_activation, dense2.output, y)
        
        # Calculate Accuracy
        predictions = [argmax(row) for row in eachrow(loss_activation.output)]
        accuracy = mean(predictions .== y)
        
        # Print progress every 1000 epochs
        if epoch % 1000 == 0
            @printf "%-10d %-15.3f %-15.3f %-20.7f\n" epoch accuracy loss optimizer.current_learning_rate
        end
        
        # Backward Pass
        backward(loss_activation, y)
        backward(dense2, loss_activation.dinputs)
        backward(activation1, dense2.dinputs)
        backward(dense1, activation1.dinputs)
        
        # Optimization
        pre_update_params(optimizer)
        update_params(optimizer, dense1)
        update_params(optimizer, dense2)
        post_update_params(optimizer)
    end
    
    println(repeat("-", 60))
    println("Training complete.")
end

# Run the main function
main()



--- Initializing Pipeline ---


MethodError: MethodError: no method matching Optimizer_SGD(; learning_rate::Float64, decay::Float64)
This method may not support any kwargs.

Closest candidates are:
  Optimizer_SGD(!Matched::Float64, !Matched::Float64) got unsupported keyword arguments "learning_rate", "decay"
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X40sZmlsZQ==.jl:109
  Optimizer_SGD(!Matched::Float64) got unsupported keyword arguments "learning_rate", "decay"
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X40sZmlsZQ==.jl:109
  Optimizer_SGD() got unsupported keyword arguments "learning_rate", "decay"
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X40sZmlsZQ==.jl:109
